# Patent Landscaping — Snorkel vs MAS (expanded + unified)

Both arms label the **full candidate set** (`candidate_all` = expanded autonomous pool 24,488
+ OOD 6,296). MAS labels are committed (`mas_ranked_scores.csv`); Snorkel labels the same set
here. Then fine-tune SciBERT for **each arm on ALL positives + ALL negatives** (labeler output
as-is). Two models: `snorkel_uni`, `mas_uni`. Then threshold calibration on gold.

Runtime → GPU (T4/L4) → Run all. Models persist to Drive (restore in later sessions, no retrain).
**Resuming a fresh session:** run cells 1–4 (fast), then cell 5 restores the models from Drive.

In [ ]:
# 1) setup
import os
%cd /content
REPO = 'https://github.com/PassionChicken-Leesuin/Patent_Landscaping_Final.git'
if not os.path.exists('/content/Patent_Landscaping_Final'):
    !git clone $REPO
%cd /content/Patent_Landscaping_Final
!git pull
!pip -q install snorkel transformers datasets scikit-learn accelerate

In [ ]:
# 2) mount Drive (persistent model store)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/patent_landscaping_outputs'
os.makedirs(DRIVE, exist_ok=True); os.makedirs('outputs', exist_ok=True)
print('Drive store:', DRIVE)

In [ ]:
# 3) data prep + build the unified candidate set (expanded pool + OOD)
!python -m scripts.build_dataset
!python -m scripts.build_candidate_all

In [ ]:
# 4) SNORKEL — label the FULL candidate set (autonomous pool + OOD together)
!python -m scripts.run_snorkel --input DataSet/processed/candidate_all.csv

MAS already labeled `candidate_all` (committed `mas_ranked_scores.csv`) — no API call here.

In [ ]:
# 5) train each arm on ALL positives + ALL negatives.  Restore from Drive if present.
import shutil
FORCE = False
EPOCHS, MAX_LEN = 4, 256

def ensure(name, run_args):
    local, dstore = f'outputs/scibert_{name}', f'{DRIVE}/scibert_{name}'
    if not FORCE and os.path.isfile(f'{local}/config.json'):
        if not os.path.isfile(f'{dstore}/config.json'):
            shutil.copytree(local, dstore, dirs_exist_ok=True)
            if os.path.isfile(f'outputs/metrics_{name}.json'): shutil.copy(f'outputs/metrics_{name}.json', DRIVE)
        print(f'[{name}] local'); return
    if not FORCE and os.path.isfile(f'{dstore}/config.json'):
        shutil.copytree(dstore, local, dirs_exist_ok=True)
        if os.path.isfile(f'{DRIVE}/metrics_{name}.json'): shutil.copy(f'{DRIVE}/metrics_{name}.json', 'outputs/')
        print(f'[{name}] restored'); return
    print(f'[{name}] training (~30 min) ...')
    get_ipython().system(f'python -m scripts.run_downstream {run_args} --epochs {EPOCHS} --max-len {MAX_LEN}')
    shutil.copytree(local, dstore, dirs_exist_ok=True)
    if os.path.isfile(f'outputs/metrics_{name}.json'): shutil.copy(f'outputs/metrics_{name}.json', DRIVE)
    print(f'[{name}] done + saved')

ensure('snorkel_uni', '--arm snorkel --unified --tag uni')
ensure('mas_uni',     '--arm mas --unified --tag uni')

In [ ]:
# 6) train/val curves
from src.downstream.plots import plot_history
for n in ['snorkel_uni', 'mas_uni']:
    try: plot_history(n)
    except FileNotFoundError: print(f'[{n}] no history')

In [ ]:
# 7) compare @0.5.  acc=(TP+TN)/all.  auc_seed_vs_hard = automate-vs-assist (the crux).
import json
def show(name):
    p = f'outputs/metrics_{name}.json'
    if not os.path.isfile(p): print(f'{name:12s} (missing)'); return
    r = json.load(open(p))
    seed = r['by_expansion_level'].get('SEED', {}).get('recall(TP rate)', float('nan'))
    print(f"{name:12s} acc={r['accuracy']:.3f} F1={r['macro_f1']:.3f} AUC={r['auc']:.3f} "
          f"AUC(vs hard)={r.get('auc_seed_vs_hard', float('nan')):.3f} "
          f"AUC(vs easy)={r.get('auc_seed_vs_easy', float('nan')):.3f} "
          f"R={r['recall']:.3f} P={r['precision']:.3f} SEED-recall={seed:.3f}")
for n in ['snorkel_uni', 'mas_uni']:
    show(n)

## 8) Threshold calibration (no retrain)
At 0.5 recall is suppressed even though AUC is high. This tunes the threshold on the
**validation** split (never gold) and re-reports on gold, with ROC/PR + score histograms.
Loads `scibert_{snorkel_uni,mas_uni}` — needs cells 3–4 to have run (rebuilds the val split).

In [ ]:
!python -m scripts.calibrate_eval

## 9) Threshold sweep — precision/recall tradeoff
Lower the threshold to trade precision for recall (not gold-tuning — just the operating-point curve). MAS AUC is high, so a lower threshold should lift SEED-recall substantially.

In [ ]:
!python -m scripts.threshold_sweep